# setting up

In [1]:
import pymaid
from tqdm import tqdm
import numpy as np

import navis

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
base_path = "~/Downloads/"

## reading and pruning seymour neurons

In [4]:
seymour_rm = pymaid.CatmaidInstance(
    server="https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/drosophila/l1/seymour/",

     api_token='a94b27c8f28f9af9be3a781ada26f99731e105db',

     http_user='imakkar',

     http_password='daunting-cynical-hurdles',
)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


In [5]:
# get neurons on the right hemisphere, and is in the brain
ns_seymour = pymaid.find_neurons(
    annotations=["mw right", "mw left"], remote_instance=seymour_rm
)
# ns_seymour = pymaid.find_neurons(
#     annotations=['BAla3/4', 'BAmd1', 'BAmv1/2v', 'BAmv3', 'BLP3/4 b', 'BLVp1','BLVp2', 'CP1d', 'CP1v', 'CP2/3v',
#                  'CP2/3d', 'DALcl1/2d', 'DALcl1/2v', 'DALcm1/2m', 'DALcm1/2v', 'DALd', 'DALl1', 'DALv2/3', 'DAMd2/3'],
#     remote_instance=seymour_rm)
len(ns_seymour)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


Get annot:   0%|          | 0/2 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/3644 [00:00<?, ?it/s]

INFO  : Found 3644 neurons matching the search parameters (pymaid)


3644

In [6]:
# not pruned
# RUN THIS
seymour_dps = navis.make_dotprops(ns_seymour / 1e3)

Dividing:   0%|          | 0/3644 [00:00<?, ?it/s]

Dotprops:   0%|          | 0/3644 [00:00<?, ?it/s]

# read landmarks and make transform

In [7]:
# just read and /12:
seymour = pd.read_csv(f"{base_path}seymour_brain_hemisphere_right.csv")
gaba = pd.read_csv(f"{base_path}gaba_brain_hemisphere_right.csv")
seymour.columns = seymour.columns.str.strip()  # remove spaces in column names
# normalise by resolution
seymour_voxel = seymour[["x", "y"]].div(3.8).round(2)
seymour_voxel.loc[:, ["z"]] = seymour[["z"]].div(58).round(2)
seymour_voxel.columns = ["x_voxel", "y_voxel", "z_voxel"]
seymour_voxel = pd.concat([seymour, seymour_voxel], axis=1)
seymour_voxel.sort_values("landmark_name", inplace=True)

# there are repeated landmark_name: only take the first one of each value
# seymour_voxel = seymour_voxel.drop_duplicates(subset=['landmark_name'], keep='first')
# add side info
# if x_voxel<14,300, then on the right; otherwise left
seymour_voxel["side"] = seymour_voxel["x_voxel"].apply(
    lambda x: "right" if x < 14300 else "left"
)
seymour_voxel["landmark_side"] = (
    seymour_voxel["landmark_name"] + "_" + seymour_voxel["side"]
)

seymour_voxel

,landmark_name,x,y,z,location_id,x_voxel,y_voxel,z_voxel,side,landmark_side
180,Antennal Lobe outer side,76078.850,57529.625,35100,64168461,20020.75,15139.38,605.17,left,Antennal Lobe outer side_left
179,Antennal Lobe outer side,30233.072,57936.430,33900,64168460,7956.07,15246.43,584.48,right,Antennal Lobe outer side_right
0,BAla12,28131.400,60461.800,44250,24827244,7403.00,15911.00,762.93,right,BAla12_right
1,BAla12,79884.550,62777.900,44500,24827324,21022.25,16520.50,767.24,left,BAla12_left
3,BAla34,77689.200,65754.500,44600,24827325,20444.53,17303.82,768.97,left,BAla34_left
...,...,...,...,...,...,...,...,...,...,...
176,TRc/DMT Tracts Crossing Point,63729.800,61567.600,60600,24839313,16771.00,16202.00,1044.83,left,TRc/DMT Tracts Crossing Point_left
157,TRdl ant,30474.100,66179.900,38400,24827322,8019.50,17415.76,662.07,right,TRdl ant_right
158,TRdl ant,75808.480,67986.560,39550,24827402,19949.60,17891.20,681.90,left,TRdl ant_left
160,TRdl post,77778.400,67524.100,41500,24827403,20468.00,17769.50,715.52,left,TRdl post_left


In [8]:
# same for gaba
# same for gaba
gaba.columns = gaba.columns.str.strip()
gaba_voxel = gaba[["x", "y", "z"]].div(12).round(2)
gaba_voxel.columns = ["x_voxel", "y_voxel", "z_voxel"]
gaba_voxel = pd.concat([gaba, gaba_voxel], axis=1)
gaba_voxel.sort_values("landmark_name", inplace=True)
# add side info
# if x_voxel<5250, then on the right; otherwise left
gaba_voxel["side"] = gaba_voxel["x_voxel"].apply(
    lambda x: "right" if x < 5250 else "left"
)
gaba_voxel["landmark_side"] = (
    gaba_voxel["landmark_name"] + "_" + gaba_voxel["side"]
)
gaba_voxel

,landmark_name,x,y,z,location_id,x_voxel,y_voxel,z_voxel,side,landmark_side
3,BAla12,37568.492,78641.430,38052.000,1289860,3130.71,6553.45,3171.00,right,BAla12_right
4,BAla34,40348.230,80862.720,38268.000,1289862,3362.35,6738.56,3189.00,right,BAla34_right
11,BAlc_ant,32795.900,74167.270,35352.000,1461025,2732.99,6180.61,2946.00,right,BAlc_ant_right
12,BAlc_post,32160.000,73506.670,35968.574,1461026,2680.00,6125.56,2997.38,right,BAlc_post_right
13,BAmd1--d,44708.380,56899.305,21612.000,1461410,3725.70,4741.61,1801.00,right,BAmd1--d_right
14,BAmd1--v,44797.770,61039.043,25056.000,1461411,3733.15,5086.59,2088.00,right,BAmd1--v_right
15,BLAl,24832.166,43726.785,38508.000,1461774,2069.35,3643.90,3209.00,right,BLAl_right
20,BLVa34,105191.080,51886.895,47604.000,1533733,8765.92,4323.91,3967.00,left,BLVa34_left
19,BLVa34,27340.977,49217.645,52692.000,1533731,2278.41,4101.47,4391.00,right,BLVa34_right
1,BLVp1,35129.190,47601.500,53508.000,1289772,2927.43,3966.79,4459.00,right,BLVp1_right


In [9]:
seymour_subset = seymour_voxel[
    seymour_voxel.landmark_side.isin(gaba_voxel.landmark_side)
]

# order in the same order as gaba.landmark_side, based on seymour_subset.landmark_side
seymour_subset = (
    seymour_subset.set_index("landmark_side")
    .loc[gaba_voxel.landmark_side]
    .reset_index()

)
seymour_subset

,landmark_side,landmark_name,x,y,z,location_id,x_voxel,y_voxel,z_voxel,side
0,BAla12_right,BAla12,28131.400,60461.800,44250,24827244,7403.00,15911.00,762.93,right
1,BAla34_right,BAla34,30266.600,62026.000,45300,24827245,7964.89,16322.63,781.03,right
2,BAlc_ant_right,BAlc_ant,24133.800,60233.800,41450,24827246,6351.00,15851.00,714.66,right
3,BAlc_post_right,BAlc_post,24595.500,59131.800,42350,24827247,6472.50,15561.00,730.17,right
4,BAmd1--d_right,BAmd1--d,30643.200,34397.600,25700,24827254,8064.00,9052.00,443.10,right
5,BAmd1--v_right,BAmd1--v,33614.800,42658.800,32350,24827253,8846.00,11226.00,557.76,right
6,BLAl_right,BLAl,14677.500,32490.000,45700,24827263,3862.50,8550.00,787.93,right
7,BLVa34_left,BLVa34,86883.000,48414.900,67150,24827355,22863.95,12740.76,1157.76,left
8,BLVa34_right,BLVa34,20732.800,38792.300,64300,24827275,5456.00,10208.50,1108.62,right
9,BLVp1_right,BLVp1,27820.300,35158.600,63000,24827277,7321.13,9252.26,1086.21,right


In [10]:
# saving to csv
# seymour_subset.to_csv(f'{base_path}seymour_brain_hemisphere_right_voxel.csv', index=False)
# gaba_voxel.to_csv(f'{base_path}gaba_brain_hemisphere_right_voxel.csv', index=False)

In [11]:
from navis.transforms.thinplate import TPStransform

In [12]:
# make transform
tr = TPStransform(
    landmarks_source=gaba_voxel[["x", "y", "z"]],
    landmarks_target=seymour_subset[["x", "y", "z"]],
)

In [13]:
# register transform
navis.transforms.registry.register_transform(
    tr, source="gaba", target="seymour", transform_type="bridging"
)

# get GABA neurons and nblast

In [14]:
gaba_rm = pymaid.CatmaidInstance(
    server="https://neurophyla.mrc-lmb.cam.ac.uk/catmaid/fibsem/",
     api_token='ffa463360b6032c9528a020209097ef0609b6243',
     http_user='imakkar',
     http_password='daunting-cynical-hurdles',
    project_id=7,
)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


In [77]:
# load some small set of neurons instead
skids = [36793875]
ns_select = pymaid.get_neurons(skids, remote_instance=gaba_rm)

gaba_select_ns_in_seymour = navis.xform_brain(
    ns_select, source="gaba", target="seymour"
)

Transform path: gaba -> seymour


In [78]:
gaba_select_dps = navis.make_dotprops(gaba_select_ns_in_seymour / 1e3)

nbl_select = navis.nblast(
    gaba_select_dps,
    seymour_dps,
)
nbl_select_rev = navis.nblast(seymour_dps, gaba_select_dps)

# make nbl longer
nbl_select_long = nbl_select.stack().reset_index()
nbl_select_long.columns = ["gaba", "seymour", "score"]
nbl_select_long.loc[:, ["direction"]] = "gaba2seymour"
# make nbl_rev longer
nbl_select_long_rev = nbl_select_rev.stack().reset_index()
nbl_select_long_rev.columns = ["seymour", "gaba", "score"]
nbl_select_long_rev.loc[:, ["direction"]] = "seymour2gaba"
# put together
nbl_select_long = pd.concat([nbl_select_long, nbl_select_long_rev], axis=0)

nbl_select_wide = nbl_select_long.pivot(
    index=["gaba", "seymour"], columns="direction", values="score"
)
nbl_select_wide.reset_index(inplace=True)
nbl_select_wide.sort_values(
    "gaba2seymour",
    ascending=False,
)

Preparing:   0%|          | 0/26 [00:00<?, ?it/s]

NBLASTing:   0%|          | [00:00<?]

Preparing:   0%|          | 0/9 [00:00<?, ?it/s]

NBLASTing:   0%|          | [00:00<?]

direction,gaba,seymour,gaba2seymour,seymour2gaba
1964,36793875,18163015,0.747046,0.685188
3471,36793875,9085808,0.689772,0.625213
3472,36793875,9091922,0.638246,0.727015
3473,36793875,9094797,0.631182,0.522853
3541,36793875,9521722,0.625537,0.596604
...,...,...,...,...
2294,36793875,3282869,-0.883979,-0.883177
2695,36793875,4990880,-0.884019,-0.882070
2142,36793875,2215447,-0.884103,-0.881902
2872,36793875,5904833,-0.884318,-0.881505


In [79]:
# ── Plot 1: sorted by gaba2seymour ────────────────────────────────────────────
gabas = gaba_select_ns_in_seymour
top_seymour_matches = nbl_select_wide.sort_values("gaba2seymour", ascending=False)
seymours = ns_seymour[
    [skid in set(top_seymour_matches.seymour.head(10)) for skid in ns_seymour.skeleton_id]
]

colours = np.random.rand(len(seymours), 3)
colours = ["#%02x%02x%02x" % tuple((c * 255).astype(int)) for c in colours]
whites = ["black"] if type(gabas) == pymaid.core.CatmaidNeuron else ["black"] * len(gabas)

fig = navis.plot3d([seymours, gabas], color=colours + whites, inline=False)
fig.update_layout(
    height=650,
    legend=dict(tracegroupgap=0, font=dict(size=12), itemsizing="constant"),
)
fig.show()


INFO  : Use the `.show()` method to plot the figure. (navis)


In [80]:
# ── Plot 2: sorted by seymour2gaba ────────────────────────────────────────────
gabas = gaba_select_ns_in_seymour
top_seymour_matches = nbl_select_wide.sort_values("seymour2gaba", ascending=False)
seymours = ns_seymour[
    [skid in set(top_seymour_matches.seymour.head(10)) for skid in ns_seymour.skeleton_id]
]

colours = np.random.rand(len(seymours), 3)
colours = ["#%02x%02x%02x" % tuple((c * 255).astype(int)) for c in colours]
whites = ["black"] if type(gabas) == pymaid.core.CatmaidNeuron else ["black"] * len(gabas)

fig = navis.plot3d([seymours, gabas], color=colours + whites, inline=False)
fig.update_layout(
    height=650,
    legend=dict(tracegroupgap=0, font=dict(size=12), itemsizing="constant"),

 
    )




INFO  : Use the `.show()` method to plot the figure. (navis)
